# GwenLand glcuda - T4 Ceiling Wave 7 (Direct Model Fetch)

Decision-grade Kaggle A/B: retained Wave 4 Q8_0 versus W8A8 with one weight scale per output row, one activation scale per token, full-K s32 accumulation, and one dequant epilogue. Target: **15,000+ prefill tok/s** on one Tesla T4.


## 1 - Configuration, pinned source trees, and T4 gate

Both arms apply retained Wave 3 and Wave 4. Only the candidate applies the Wave 7 W8PC scale-contract patch. Rejected Wave 5 and Wave 6 kernels are absent, and structural checks prove the retained attention, grid64, and r256 bodies are unchanged.


In [ ]:
import base64
import datetime as dt
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys
import time

REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
WAVE3_PATCH_SHA256 = "5f09f6147636c36db4e23b9d5f16a3384ca4c0b69679d5443d7c9a396387a508"
WAVE3_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19eXPbSJLv//oU1dqwmzRJiDh4ia3Zlm11j8NHeyTt80zoaSmQBEm0cNAAKInb44n3Id4nfJ/k5VG4CFAG17MRzVE7ukUKqEpUZWX+8qgScmrPZqLVmtuRMI/mzmQ1NY/CYHJ0awWe5YTy0ih0ex1lGT2IcYVGB7Y3tR5Ee2zqE9VUlP50PB1Mp0Jtt7uGcdBqtSo966DRaFR73o8/ipbWbfZEg37++OOBODoS1p0VrMW9GSzhRygCqxVY5tT25iJaWOLsw+Wb8zNhTiL7zoxs3xOhY47FLPBd8U4T8LsdhcK/90SLqM38gPrNzcgSP5+9f98Umt7tE/1QPAhNG4i3L8WJ+EfHaIv3L4U/QzpRYM5m9kQsrUA45sqbLBJqZtzHNaPAflDEqeOImCD0NsXY8Se3IlyYgUXPDk0Xvvi3lhc2RejjtYMWUAPirdsWt85NyJ5YotY1RODf4xh1TeAINXhoXZjelKYIt2e6dtDYgY7RJzp6GR2cXbbvxHSsUOAMPP9eXFye/nz2Wrz5IC7+fHoO396fvf/l/G/AbngC8kg+3JsSoYnvhSvXmorxOrOcx+I04es08IFb/Yem+OmnD7QuoZj74vX56fvW2F95UwUJkYToJCF6RkLw31/6o7a48E/F51CEEUiIK95cAGftENZr7a8iUfuE82655q+waCcn4uV/XsLAHL5QVySl16LWf+jDsuhaXcT/TmA9eb2Aa7C0lj1fRMRF7vYRGB4tUCxB8My5a3nAyNo88FfLN6+ht2N61pHRFJE9l789M+rHtOZCmG24FrqWOzKvalFbNITsWH8BC90QC9OZvVC78A26vzCEojSMayFqvmeJFYySlvxRMrDOFcjw/MdI5tOVlyFwfXU7fnQookhm2m6KqQqk/taKbMdCXgkYVSMeUxNZHwrtBRBq8kdDBTLRvU+sT1a7ozZVVTTwQ4vX27HmplNXxEVkzhEHYI3nKzOYgoBFPjwKxKVf82DB6rRGpGOeD7IAbXGRQjG21r6UTVTIpTklQAE5BYGcRCxsuDzn1ueVHVi0oscCxegZCABID8zP9uAX4Av/hs8TP5yg/tSQZrDyPFCEyWLl3YZEy4zEx/Ozn968ezd6eXr56s8CG9eH4mH0OTx6GEkVI910V2EEgxSn79798ur0EhRstYS50VJXHdJQWOZkIdatV5enYuKYLigYKLjlzJBLZkTEXB8eJBGhKe4XuFY7jAcZmON3zXoADnInBAtkN/AGlV7cB3YUWR6oGnLjHaHoMQiZPRUTy3ZqOJUjUD/xAphpT49ADOqIyJ2uYOUKuecFAun0WGhtArACwjWoy8sifCk05dyDa1uf3ORB0bIe4UIVxoLEzlIWg40BvAc16BotAguyRAo9R1mfqCimKBh+YIPUmo60I0wnnpPe7mk7zgnXoPXP+3cglDs7tMcgCAqIGGD13BnNLdcdua45+tyvHZDCK0szAIxVViA8y9H957BJ6trTEZx7RlM1SFupaWDNhTIGsXwW/GD0/zTMXp7h5dkPupa/PAayz4LpD4YB11t0neznVCimY889ASikjPsAeAB3KAjXQ2x0hCvFUJ0xlm/B7kRWeNCQTT6Zd5bQj4Vnok2C59lhZAWhmJgewISAB5i8VN7KHVv40AVaR9MBo0a8V5hUdmKj+XoUgQbyN28cfwNFHeZa88Tw1oPfjL+G6de1P0wG+uHsr5ew6LFNOZIqGfswoT+LwJBaraW9tBzbg4HethzfXxbHN0Yaba8pv6leblC0CPfhhBrA58btJbLg2dID3Y6vb18MlOCti0EKi2gLfZd2hF5UCTkjpvYQXoHYMzVJDodaUIIDwXScKYslSSXwU22KKxLO6yFJZ7/TBOM10JtaN5HOtA+to0ZdbO96WHpbp9sICtigIFGuf8cu3gJw9XvwRKzABk0HREADSJ4prg/gNEAwA4MCgD42QzA/CTUkgCgjncQW+eHIMR/6oaCiNBKio9EihHLNWxAM8q0SMrYHjtBqQnwaWw4OAEhaD8A+Z52HIny8Y7VogKkRTMbTRuwywQo4kb0EYACvFs0cDgPIBBYIpCX69diZhWeARpG1zdiGhFy6fK1NwwvK6JowbhGu0C20Qeiljw1CO0f7btqOALSVgwN+y4VJ9G8CTYCtUlDDhVOqoPitO0wcPDnLEyF7gyHoGvIRK0e5t6eWfE5Gc1OK2lAyPOqDkONS+7NZaEVMAeaohCyQejP9yYSScQaZiTB8gCh2MkOMx0kizw52SL42PXH7YMONwfIDsvxJISnMQ5I2zDwXNY+f/cgEjWb6k6kMt49svTEytXxQ6yw4lgwKrPdyFT02qk4z/ZnD2HA1VsKE7SB8TdbxeFTx4M2HYrPka9LK9mQrPXe/aww315DgEBwGDokoSgGhPmgw5qRSjYwEh0R5kGiUiglNxiiREKKP4dXInnInUFOpA13ZSVeHJZ0wLiGYVI0BWnG115Mx1gY7NVqtab/Jv8ixTe4ixN14gTWVZLg33NY9/siMBQYxd/wxAhK49uCXRwGgzPTOhLAyhJhPO7od19kdSBVbo3FoHeRFKmuOHwN6ctvoZ8eSvdnPDTVlvzZoklHb1m/AvYebGsrWjOcAnWASMxsAvM7c7XabA+DuwICPHHdZdohvukY/1S2kH8JJnrA0gDIWpWSJbHp+dvpagIGxIFiIHQkEcIrI3Bf9TKCobLJWJ0FWtwBRHN++EJh3KDA+7WwUOJR2NfoFBlA/2bu72VOGnjzSzU68ILpe4BqExeKUghHJNheTBABH6FbXM3jxbcT0vlGPFTXhoRGzQSuIEd+Ta22UPghXuvRRGCOzPIE/A16NBgon3ZqCGLcRpETZP3jM7VjU3sqEDXka9Yxj8zGwXUu8+o/z87MP21zR2/FJW9zZJpl+clkTl1oRlwsrIWY9LCGYsRGqI0y/3VrWkh1taGxO162QLrOHNIWn2PAQCulhlY4pnE2dGyDPLiB4pzMLXEkYze24ocogFgLNEMY2Ac8kjPNPcD/jN+T94gTEc3fUjTtZR7k9a8t/JQ3UsgZgmpbKPOJVIXeaRMNIn/Hjd3xdjCGIfv/+dPT23S+/fIxvPluqKjqkjJFMJpnAFfq76vVX26pJ24ba3docPPpnC1W21PPNEOhhnsoMG8VRw0J9lJQmSRmPk1KJlLa5FoHWbSZTLd7sFcKaZClmWlOOsXBHz8Q6Bymzj6UC01pJd2XJQM9OiiHVGGYwSNbp0/mbyzMOMjRVbao6KGRn0FQ7oJDUgDKmo5en58cJUmO0TblaguuJ74IXY4laeGsvlwDc9wsf/bw1+jctf9YKTA/MCaWV64Ij9VhkgJOJwFz87cMrDE1acWjCycoE+hsyZjoGPTMxDvvlw6uzJqjsKswmaF3O21HcIO3ChjTRosRSlwWUl5QnFG1RuxVtRQEegLkhfKlvIdTLieQmIRUJqV1F0dUigYKgtmJfJJErkoFFqcsD63/lTdqY4bwuJ50V3BLSOsvrVtIqk07hlLFqW2Q/5lCK8Sre4pisggBXjliobBoroxfLppqFGWcTZrgdSm+sfwQ0j8MJuFzXFTpkMAUzMVs75BerUaD9VWQpJZgs0dcJJviS+kvnK89DTrO71AKb64P4o8kF+4V6AexEGyQVos1Z1dt4NTZMrd4pcRoSmTilBSdTzlBhaLTbNdDI0S4ARR4q5AjUTeXf1P1N14iHBD/Big4Lvk5yF5yXooNCEQP+1LU0IcKsipXXIPbrnettDTqyAdsb3uSjeevqLvPWNuat7eG8+zxvfZd56xvz1vdv3nqb593ZZd7GxryNPZy3xvPu7TLvzsa8O3s4b8Y1fSdc627Mu7uH82ZcM3bCtd7GvHt7OO8+haAGBM5qu2ziyWxS7/cM3UzcLqtRPhiT0bhJWxeuZUWhwLRy7A7RVgyac86o+NAT9xItaX3HZqCEa29SjOH+p+OGzcyWSrkb+tC1zXCeZiAD7WJPzgnjh1ZClxOq+KExxw1A1D5wvNNNJY1iEMlg15xiJiaXUuLwhX7EmbDNrGycdmoKozgKvjlVOW+HAsTpoih2x5A/V3TrGprM1PYjLRoGt0mc1qTNnZYn9BtS4rZfhixKn9TjgmS34znGc8tHcGo75YEueQjr1EGx7RpNY5BwUf09sVCrwEK9Egs1bvtl+I2ci2NfUPQYp17/8uFs+DtimlGBaZ1KTDO47VNgWrcC03qVmNbltk+Baf0KTBtUYlqf2z4BpmkVzIJWySxQQg7aPgWmVTAEWiVDoGnc9ikwrYIh0CoZAs3gtl8ocxRPVToKgRVJh4JPP3XUQeqUyXa458sZVnMS+GHI5+foUFdyRNEUHVWTR8fo2CCedeazrKE8cxrykcV7PBKGxztwY8ifCU7xRrZrhYp4Jc9S0NnAZ/2TEzoX+EzX6BsexfjhBJ7bJFL5s37yhFPxHKUifj77cHZOJ/9qlmtHI9ySUZbrevb8sq61Zr4zFSsv8B0Hz4wsA//OHDuYT5uvHPDTxXly2uofEB3yHtdAU/FsW+tP4h98gPm/S0jVdVVLKOH8+JjE0cV7UftHp/1M+JPJaml6k3V9SGHF2PImC3GF59tay4UZWuNrcXr0UkytCUglsH1h4QkXeVIUBuJZUWtsmRHvW/U5hOMD7XjYHC++/MRHltOjzP/8Q3okbgPapu8YnYK4ff30l3j89JfYcvrra0fx+uogPf1FpyUfO4v3CCFV0/p8qK+M0O7nyNS2ZuQG9k0HyTqDHu7hdw29OShhfSYfTKIJAxj+sUX/dLfoe5pKx2K7vVRTs5uNm1ZQo9kZtKeTYywdxP/agRo9TdzE+ySyHad2oGG8mZXywOBHlsgRH1ciERI1KbF0mpFmWS8Rq5RWXqwSSqC+ZZQ2WC/J8M/OcFPIEEfiA1+lPVk6jJgdYODjPBaw4wrvXDPDjKE8u9yjVerrm9m70/YfC/V7WqgOnzIfDAoLpf6xUL+nheqSh9JXO4WF0v5YqN/RQvX5j+r6/W/aiG7v4VZ0nzLojUH7W7aiwbncw5n3eObat2xGq+093I7uD3jmxrdsR6vtPdyQHqg88+63bEir7T3ckh4wwg3637Ilrbb3cFN6wAinttvfsiuttvdwX3rQk1PfCeP6han393DqAzn1nUBuUJj6YP+mDn6InPtOMKe2C6dq2/s4eV1OfjdfruDMqeo+Tl5inboT1qkFf07V9nHyEu3UndBOLbh0qr6Pk5d4p+6Ed2rBq1P30KsDXZWT3w3wCo6d2tnHyUvAU3cDvIJvp3b3cfIS8LTdAK/g3am9fZy8BDxtN8Ar+Hdqfx8nLwFP2w3wCh6euo8eniYBT9sJ8LSCh6fto4enScDTdgI8reDhafvo4WkS8PTdEnYFD0/bRw9Pk4Cn7wR4WsHD0/bRw9Mk4Ok7AZ5W8PC0ffTwdAl4+m6AV/DwtH308HQJePpugFfw8LR99PB0CXjGboBX8PC0ffTwdAl4xm6AV/DwtH308HQJeMZugFfw8LR99PAMCXjGToCnFzw8fR89PEMCnrET4OkFD0/fRw/P6KtNPKDXUDtar6lpxtP6+6Z/9pnsT7+3v5nYiz9p+vTH3zT9K3BtL/6o6dMff9X0L8C1/fizpk9//F3TvwLX/kf+sOlfnmsVrIFWyRrQawWg7ZPgWgVroFWyBvTSTGj7FLimV7AGeiVrQEOHtk+CaxWsgV7JGtCToe2T4FoFa6BXsgb0x07Q9klwrYI10CtZA0qEQNsnwbUK1kCvZA30Prd9ClwzKlgDo5I1MNrc9klwrYI1MCpZA0Pjtk+CaxWsgVHJGhgGt30SXKtgDYxK1sDoctsnwbUK1sCoZA2MPrd9ClzrVLAGnUrWoNPmtk+CaxWsQaeSNeho3PZJcK2CNehUsgYdg9s+Ca5VsAadStag0+W2T4JrFaxBp5I16PS57VPgWreCNehWsgbdNrd9ElyrYA26laxBV+O2T4JrFaxBt5I16Brc9klwrYI16FayBt0ut30SXKtgDbqVrEG3z22fAtd6FaxBr5I16LW57ZPgWgVr0KtkDXoat03eNC3f6lb2ysepPZuJVmtuR8IsK2/v+lMlCMV4+70DqrcltJk+0bq6oljjabc96wi13e4axgGe7XuE8kGj0XiUOr0ksEtvZu/KF7MvV2PBVUHFW2p8YUXit/g44VFcqdT3LHzDIr5dEiu6mxEVCxKzwHfFzc/vXv3H69PRudbp3gyxpmja/eomoXp8jK+IHFmeOXas6c21fFk7l/Ia+77TjKviHIlP/JZF+UpFN/dSxXDhr5ypmOBb3/EFlSKtrRxivWt8hbzWep0Sk2WVM8PPDfvn8zevtddy4I3SgWNhVm2aHTq148vx4Gk2sxFI7bFkZXIttJ3VCAS+cCPwl1Z6kf+MS6NigCosEJ30tN2lU1wa/GfPiHnZS/jPWga2Fzned7XDKxaFa26HddFs4Be+gVLIuYga3KEVjmvY40tJ64f1YUr0C882V9kWSyNP7akZWVjRzQ4zpdvMwI4WrhXZk3jFkP1Yq2qKlSdzxAJr6ZhYynFriVxePC4Bdw+EkyK5b+jt/+swR89fRi0QgZUHAsOF3AN/Kivezh1+t+ccS8rBg+mOhRW6xaWhpGQc4DKvrDgRrmsqdjgKfdeq1cXz5/DM6fGx5d0dH9+ZwcgPa4c5MTqsp82HKU1YKknyt/TitsUC4ZV1fqkiedmyHWaJf0mX6pfbWqmoEOC6ZjN/JaC3vubGw6PcaCeFGkBk5VhgFqLRDBYFmVo7nDt4cwSweVj/90K/VPC3dY5bbKPAGrKtN95NetJ7ODtU0bnT7TVV7XH1wcL3CrLgIOZiqv0XjDEx68PV2EVQd3KVm7Nog/ZAyuq/A9B8XtlBWl+cCziH7qjXkTOhgs0bAITUqFayd2cHvofv/JRSiRg980QehWrPcQJ1fLUt4k9WsGhm3JovfokLyeJYXppYG3HK8nTztyt8n28TXw18DfL+V/mr7V2LH8WnK7hMv/zn5U38Nts3Hy77KbVcWcXae01RxaUZ3oqXTZ5wo66IC9O1QDWtAGYdCjPMmocLGCsALJaT/9wftUF1zJtr8f/+z/+lZwGrW675KzzgL3BTXPin4nMo+BXIVILawfdsrnnpwS0Em9bo9GPbVr72NGpUrzD7xmSyC1g8uS9q+GTHMUH1J8ulWECLVuS38BONz701TQklqCnfy4yFw+5lsfhYeiaW7cD1upJ0i2vxHSWiIm6A06Op7YpnMIKTE9G+aYob25OXwCmJr1F17x9OQBBvaKQprWDleTABelN0iCb65uP52U9v3r0bvTy9fPXnm3qTxQ5f8XxzdBO/5PkmfctzSgvLeNKjbvi1zyAv/ILqfNFvYJb1EIETSK/hNQOL3nvaFB7WDEypYXEWEBQQhbMg8MEFAjxEDkllwLco+1FcfhDXnjDa91IF2plPiaXB3o9MufJMU2qPTDkz01OCBUZwM4hsxCxsP7ahc1qU/Sh+OFWAR/dvaXlY6jRGmleXp+HOfCso2MIMR2AAEq/r366QA/e1iWMvl+vj48j3R67prUdmMF8h9IT1a24Zgw9qKFAALa2xuvX7+Ccp3bYsOFOubIRP51YInPyhBgL4s0Mz+VMWiKfWeDUfmSEY/mhkff6ullnipgCX/zDzcOC2lIRNQcj5LAWaGRF5hCaX2NS11ttcVXUk3Soj/V0tVccNmuSfkhq64NfAGNmLBZCEkCZ9d3U9Z8pLaP9p+2jRC7dMII5GKCGZ4wM6M7VRU8yaYlQHfEfTkDexClDduAKyAtBV4+rU3XYHCwt1NV0WL91uUfHfc3cVCRtAXrzAb6Pk22R059ubngW19qq0vs6sQPwC8AVqXgiz6nMVVPomi78zrwmiSFER2mWFxoQOulsKG+3aC+BQDZEaxOQulkAs1Q5CCzGpCh/oKie/wKLQ4Omt4GE954V+w+Cke7vdSZbeMfrK5ISQZzwkFEtFUYCzAuBgPZiTyFkLNePdZuecd/2AA/kL29iRXGZXgVlU3+ycYVb+Tnvj9wwbM3fqqVeWgNlHfB+/eCmNbotCFXJjjjPVoSey2gFCeM69YO25uY5fCTZAR1FtG0azo6JcA6iCNxNGYVakMwjy8fKvo4v3vY7ikgMV1r7/7fu6MgGbESGyFW9/SW9nVTLW7qQDjtgEV6R2qEQAv6BT5DkdVu2FMQqWO+OX3FtTxe17/Vu1q4BsQTMHUzFK2Kf/0FXOAk4a0GUijsAamyF6Mljs15tjiARSFPll4biSF1UUfemUxQgYl7RACwddwO6syQ9xVyGHcGBpVTCVMkzPB2IyZRAFa9CfeO55+VFmYDWBd3d2aINjLBRuDtFBZtmpPEXtsL7R1XpYWpOodkjjpn45OMYR4Gv4RzKQPRHP4zFcKUo6tuthyahL+qQ9FCXbJyNlmcclknT4bAJLDbBwmJE3tV5OIPPsxwm0iwTyj89ImO3F+TxMwI3mazAQBAcFaaIqfgJXE2zS0rQDctOnvwKAgU+D695v0duKpbBSoZOxD4G9zFXlqJGMBBYOg4zcHawWIF0YYRVARE0pi0jX88lmmwE/ltqESimPCrp6uJkDzDIKyy3oWhm3vivRxtKcJLMpN7VTYUhOnJKfR1UrkFnEDKA3c+xJ1JoFlkXojrrHQSbwwTWXOWI0fzNaBaxdzNqkm3QYpuhSgvtJVROpwjGzSFwurBy1yHKXETbQuzxCHltoRnY4sy2seWJTem6J8XcQrZUia8o4g2U89HYvZkfVLlz5o6zP9sVM326d0xlYSWNTwdH8jE0MmU7E/7Imx8eYZhpNzKU5saN1Lb/yyFIqhDECnAShbCtKfzOnQ2y35/K2sXmbK1TC85TlKlzUarWEHr0MnIpl4Ne6OMIfzzZkbyMjl/nKRCEaj0YrCGwxS5BLQvF9CBVWy1p9G/pgE8fykFvoKdMKJDLH+hj5K5AGrgqPHSh2PtzkEso0c4BzlAUWoWSN+J3l2PAKtx9gicBr0IzrMqZlxllL3p3eyNKpy5DhcX7lZP1lBnooiQnWyDNdUJaNiuykY8u4XPvtuKHKaqR5A+hHJqkOemCeNJlxvXZU3zQNEnG258EOqcPYDALbCqoC1qPFfLIyr21Z6jyHyk0rxwfxI79eGX7TvlL/eCT5W1rG1Stxc0qQ9fv/3f6+Xrlt8D3M/fDV+bufxD3l8saIg7+C/YClHa/FMnoww1yoBLE1wsEF5m4d27PqGY9DwQthrQ6+BUTHAUTZ8J38jr9jdOXU/y6+czDza4YT2wZ/D+R3224Qp2kKG0HJZbkHNB4P9IFqKEq7PxkY2mzrHlDasbD9k97i4vL0Egn+gAsc1d+N7kcg1rUDcrEpl29zqMHBsM+7DN+jFb8zPSy4SzabQ4xcO/wF62SFtu8pTO8T58diJUoTJmnCDaOYpqwaJkhYuNUNxhk17wjiixtK53C2RF7FGkA3Q1DWyAZoskwvZOcTgACdTg/UGrUKldbxQ07zyzILruX6wRoLlB2JGpUaowpUwpyj7ERC09sY0twv7AkaYowIOA1Aw714r2BJMujKxcXyfXVd6z/el/J/NJXXlmOPSZjAZoP3kpQiAwMcuIo49dCHtmYze2IjfLA5B9xIBNr27uCGJVkDlHjPzPSQI+EKJjtEsTaz2yBYzAyzmJJfwBxyBUDq76nEsjkGt53lZaBhiIQfwKNEYNwRevO1g83CGBg7IYuXVlgnnyJNn08CUzo3uYxkUumuSIudHRA6lJUWFvbCUQKe9rN1vygDfG+uKRNPJdZCJU8LZn+ryOQXb+BgR9TUOLkHpqPWr5fZm5LNrmPyQuWeFAQPJFlsCcG3SjciS4lFCzD083QHC1kkgwSKuIDUwxEnKI/WJC+vLk/zW2YZasBcgL87mRbwAxtcOgh/cTUdq0WpAbQ3Yx8iMXD6WiA96PwppcTeouGjcnbmQibEeV8WaUf3Pimk6VD8RrW5iemRXz4yE91QEO7/so44RicqxBzzAZ8RR38gwiCO30+BNG4wF6nR8m3se5SuldyDXwUedsjmKkubxpkQ0NTPIIH34aRJlQ/xJ69AU6ybUj5tD/DJa5ZT2nQ1NjbkMmyhN/N8ImDAhcG8BSfkQXbvUa1pzULgWuatPXkapXQ5dkU3AldETP0k70NpQYTBY4Zcy4Rnc/KEq7S3dUzt6Sp+PK7fjzz/0ry1JPZimgBQxBMUZAQ2jQJQaMXZKzMCUMN8HyWpSomBpKwJ0oFXFAjRwMnKcL1LthtoA4FzjP5z3yJZbJXTixPprGoxNq9Q3RDpJYCDxG+YhxJqMcpLg0i2A8eBuMuV6Wc2DNHFb5smY+fBoSnZNrjG1wfH1qni6ErXFUY3M13bsSGkPsR8JpgNh7ABoiq45TiUhI8tf7rvX0ous9c2tgjM10sLhb/06bIgTQ/ls9/vNdV+knr+ebl6708tZzPzjNX3ItrSMUPmW8xOO2QFwz0bf4auHCI6FntcUbz862o6t4qG4x68HXCN/BlmGH4rjhK9Rc8DL5ES2Cs8lVS2Lu/8OSUmk61CTAsT/M183IORzHPZtieDgihB2W6TNDC3jgXcDtaxCZwsMHk4ze1QS0+NbX251MCDKU/HrKNNYRY6cJKNBwxTPCoPq8QnGCBcAZsONrWUHtlHOWPSXRQxOaR1Cw1aDbz0OCJ6p4mFLf09s5TeBDALoiTw6iZRk7a7ZtY9kJSGE/e+wvqQ8Yd1WqYbS5iHC4ZsIYCDdYM1/q7EvICTcKtkzypJxwGvLIHTNc/baoNK2fxFQHBpbesCSFjsMSwXN8xVgC8EY/97GB3DhMFaNIV7LJ6jVpjR3+EhrbKHYMrHGtHCXoXRtWicSHnka7XnrnJfh4haXiQhpYvAc49y/MPy0X8j4WQ56iUzTum75iQh77G6dQ2g6irxLmDmktzq4yslZMuYi761Y65BsMAa0/kJ+g08RfB5atvCOdouOFoi8q1zAd3GDRnSdTqTbteEkG5qdLu9cfmxvs2uuaBu8yYd6DNUOtEHH1rOTZeZbxm/j6bW5xUgGkjMFLf3wCaAJaWpxU4LHYE5Stze49Ljdx5iyxLjrey5u7xvG4M9U/MxAsnuW+GWFbm/IQIxKi2XwM7YQwQKcujNYI0e2MRZTS0mB6EkwhwE+k7IDtRSnpzxfK+19AEdWv6shV4riAHW9sqdArN8Qk2Ah3+7QnZeHzQ2GCYB4et8a8RqSTmDGjuUt3VMHMyXK2jCah87pl+kCjHspEFJBhcyx8Eu3r75eAzh9B2W9saNmHt5ggj3+SFa5IM3LTx4I88D5FJw/Mxhcv4nFvJ0L4/VhAuA1ykDl/auqV2JLV0j/tKhL9k8Tg2TdbjxrbXb5dchSM9epz1BLLaW7ZHL9SXrgHJRe84sfc7nk4rDbqIXElr1dJp00ulIei8cjHJcY3lztPupeV15ID3Ikb/cW56mdFptpfNS1BA5cIRMhbasXGtq47kKow8wKAN4UrxOu6kaoHk9+NT/e5qHDzEdOiGQpCzG6FRlRHZsLcw7218FtPSmmJtLGDjWE08dbkrz01k2pChVdQYqwocu6T7+ugJpQcGAy/gUiBcgBlt5eBwStNGLOLqT6X1wK+R0WzFLyW/MZmnooJMPfOJ4lvYPgd03pXvHN3yE0/OZYLh0AE3pHO9bgckNOpVFPkt8zOeGQ8Ok/PoC1MDF9HOSbkL9YHLwWPAA+CBMrxu7uHS+B8/F3BzdrJY3TSzxQtem/r0H13z4/zNc1vjqLfx6d9NkisB8Uxjt1sV7PCIqTpO0SpLaiN1D3NVnig2giF5Tt/sMBykbMj1wR6Etj6bBo3H9O3KG71HN0zgndmAj378FM/FM5pyQxQ8tCqJzKxFYLm6RgX/sBxGI2jEyYMvMmdLm9FFIcizYnLxMhci0iU165YcSk8FXvCkeC7jZMAtDnCAgIjY4AWPN2D/NpMGYGqXhucuJmkP4JM0hUzBg400Qp5a/TKxFVmLjo1VgYMwlRhu4LhDxWROT0xImJTFvsucDIHSKjxAMWa0oicoE5dk63BXn7S93GYUKpU9k0l+m9eF5EIZbwcTGKeLBUV5WeQhILmfcj05umHyOZ7aCxmj+Huk+lEdBMNMSySWQ3jll4eTOO1nVlkwiSbvZQrsZQ0oAzILJQo/zs9N3o4s/n348uzgWVzWJ+dkPcLINPCF6xe5CjWCcftABEHLWP9ODfAyafj1GcWMdBI0GAAzknwXUVHzv10bPW+p5dwwyKPU2PgLqmQE4BDgreXhGioKLMeDB/weFdsE857AAAA=="""
WAVE4_PATCH_SHA256 = "8fd9de6b6e2a41b84e73835530b3018bf037e621a6110737bbdeb26139021bab"
WAVE4_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1be3fbNpb/358C8Z40VEXRIkW9k05ebjeT5nFst8kejw8NkZDEEUUyBCXb03rPfIj5hPtJ9l4ApEiKUuKJp+3OWZ+mtAngArjPH+4FPX86Ja3WzE8JPZoF7sqjR+yaLuOA8aMJC925kXAy2dl04IceuyZuhw3brm0YQ9O0aHtIzHa7Z9sHrVZrD92DZrO5j/bTp6RldnpdvU+a6gmvpiFZUj/UGqT1HTlhfBWkj7WGTp5H14+9m5Dw1BuNWJJEyWh0jI/vviO/HJDsJ2ApiZNowsgT8itPo3hEVh3r12IX/FkYNE1DJ4muuCO6a+V2/PkG162TTzpZuDpZw79oleokdOaMelwn+HA8f0koxzl0EkfqLdBkibNY6wetKlExiKeJ7zGdcJcG8AjTaJETwTXrB807j1vSa4ezTxU6JRqNzZ+3483vR0fkA02WhK1ZckPWNPFpmBLtz6/OSJO41J0zDvyPgxUn6ZyRhNGALFgSssBQMuzbUobi+fUyvD2oysljbuQxIa6KoO5dSJ9jdFk0dxXLZvGNPxVEgJsw+A0YRhKF/t+YJloVd4fKQob3YyHTKCEO8UPSNgw/ZWD/9baxm+e/m3F8vVlUDKIog9vPiEMIw7Y7utkGaYhfzHsQB1hfCtOsZvN4lRrkRRTyFO0voCuYn0RglLhJMMTQZQFHwaEVeuDZWQKulHGjRAz4DP/8ECRLoumUgz9MIzHEna/CBVn6nhcwwiNp2p4TMFwjveHkv63hwKi40og7MAKcKdJtEk0T3D0iVoN8S+yG4GvPHm8kg6O4H84C5rg0pq6f3sBobUI5g/HZaPjVbCihjMu6iY4Ld3luCgHCf+2Le3TfZr1u4i7/Kf007+SysXeZOfWe+u6K2e0JL2F3+/fiJaqeQq261mF8qUT+RVK5u2S+UjpbEqpIaaekvBo4xhP3SEZTrl4ZcXq9wU317QqY0WnH6/d6AMz69mAwtGuB2Q4KJXS2ow+qltXTe6QJ/we/9/TpATFSmszQyJdOvw1/Us9LGOcOh12Sng1hqwlu6F3IlAtrYYNH+Jwm+GCzJQtB4jFNUj/1oxBeTm6Eg4oTNvWDgIAmQRdoEpTUskAPwY9xN0oKjmXaseQrDsoaBNGVJNbpkeeAU7yVi1SgR0JTwJwHTYNdgwKHxFCrMWjgz0JiE2MywA3lOnx+AS7tAP1p6/5+BL1Z4ADHHFj5iNyc+xek+YRcw9Mg5EfBrxGZgVISl/mBFh5Z3V6DXBN4YJhAQzDufVlCyv2urQ9Azv3uQB+iA3n984cT5+W7t8cjMWEhWohggFH1PL1AR65jNAGukVSKzuMYcgDwu2kgJTv1E54eqPCkRjZNSRNHcqL5KSfRVQjTClJX4AWuEh/JFdTDZR7YZ8bHxdrBLhKjKG0bwxNDn3jFrn2ekskK1pUwEiLARfjqkRZMveI0aBgHLaBVYryWYxgMVsh80xrkzCendMkyZY4DGqK7gtUJOlHiz/wQ4LFm9shr/3mmm80dGgnTN+80vU7AddOl72YrmNykQP+JIFO2jW9tMa1RNRlfovgAjRiYUxCrQBY+F7QkM4Vg+TxKxJlqGYOIwogEUTgTjOQsWTNit4e9jREm6hjx4uyZVNTTlM4YsVzDHJH3lHNigusFacA61hbRwP7Jh2cn7wm4dzD1GxScTtwI/DEHacNCQiYNR/pqeBMBI2BvfD4NWmnCmOQrAzgResRjgT9hwF0Gqvfy1c/HJz8cn5IprF5ISRASLr4F3IVNS+9C6ASAlk6u5j5grgVjseSSZHsL1tZCpVybxItS0B4kJa2mb+pmF82mDw7SrtoN/hjg6eiSGACTSOwo/dcrbeDHYqcc53Z0yGKWjINZ+1S0izAmw11Ng15uWamWonYc4BlRRXcjYTPojFr2MH48+G5cfD2BwQ+Tx3blNc72cPq4Y1V6w9YfJt5j24b3ct07HDAYxrnZ6wzsi7FAs1KvzgGAoJpd7B8M75LzTg+HisFXNIllnKEBaswkAdVxKXoiubrAk8wQsoEFgic7j51PF+P6Zks0Ly7GSvbDtoiM/aE69lYlvxkvuNURw4tC3p5IdLTzjkrYW/0kn+VyhWwvFAKvUuqpLkUhI7WD7LhwJqxAOW+wH3TD0jcCkg9nKbilPZ5fnRaW0TqbEHj00E2p7xk3ik2Ddk8EloFpbgcW/OHzRI0ewuAEltwdb+GtTJy+JwdBDDW4GGS2cVRfJx1zXCUIU6oeWySBYOggSS6VarMHE7MnQhXHNatQnk6cZzSpiI0tCh0908eaaTdhoEykWbuMHJKMS0TyOLBZjiTA54GyTruvKxWwxjv2sQkL3xKEadFUA9VSSymwuKOrBSHV8c79qLAmV6N0zF2n1EgjYxZEExpkptTVhb2NN1Q+Edlj36ieGGUVRi2yUULVhm2RiLKGIHFhkc/Ozt6eOKcv3p0cO2+PP56BxhVfFZRwQhMB10kbraOlyCM8klEL0BqPpumSXosjeearwDFtzONC9JfM2zu4LLxaIrldSVMHfrWn338/aOPPuCrLlh9Oq5ZoCWn1xtmO3zz7OBJc6rT7HeQSPIdFLr376awA98BpiHM4JzSLkTJuaGgzrVXoQ6xfNgRCwEjJaAIBN4FzAQxRrehQFHYMQ4ibHni0IHIXQBLYnfgsMcgxbBxxWooHLdRkBASCL4gh4Wx4Dqd/AR/lugBwRRMEHnQSMCV8ARRFSgPOLJQXp2thDqsFmGAJuCxlYqmITwRci1E2EwZwgKT+knkK3KnMC5hlzIRP3EZ7OdKTAO5LCVXAWz3Bez90GGuf+8gtA05VAM3wBFJ7Vi8jFcAoUl0GXQxznfbQ3KEuX4tkyD8PWGqJo+YqJCN3MLQwAsFzCDvZuYMSsBneO7CJvwrYxL8dsOmY4EZR4mZ7uEfif2B0U9d3IPuCatwr/pEMs0yhYKbVlxCnlmF/WJwTfz3Oif8f5/x2OKdjWaaI4JY12Bjoe+fUdE6Oz5S2QShWsV80nL5+9X705cgm/j+DbN5LZPO51Ooy8kql7u02lVJlE6/XnnYNo98Zumb/8ylVNXpnOlW1C4DaxZpREx6WEFu8mhAXCz7k/dlH5/RNvzsi34BTBD/jh26w8hi6yAfaoSTr8GW/i3nZwwZsXg58/uO7F69FjRsGWV3kypEs6MZC08n//P0fMrE0Y9GSYfzHJI2AHCFrCTCXpTd+OH7zs5ERxqRMRhdrNE0k+320SirRR9SFEbTlQYjwIEoNcgZTwLayPBHHHJhMV/FIUsNFcARKG8sK6E2E6boAM4oMCxicrCEagmOeiv4TASBnIr2HSUZ/Nk8lNeHjZPYPUJncBCqIc/LuwykYxsufXpy9evfWef5fZ8en+c56IhhMQ5FvdTx/rYUjWX7wxFOUT7CvyoqEBvRxRHIWXBlm/eXsL8vQbsmWkWL1VlZ7JNiVp65F5kuFHNiWJJdX26f+Nab5cNet7Yw2OWHpKgkReF6+BaqXZEkXwGGK2TzBN0nubyyJjtBApwHIOXeGNIOoog5EgOacoW4gLA0BxXsskcn7yQqrjcBWYNQGO8q9OkKoWtnPbnj3LsYVP4Y/sb4kPAiIsuKVnzwh7axVOS/YFsEtqdB/q7x+OV2VDzDAybgLWMxyFWh24081LeDmtZ36AKHgVib8VekCdBnFGHkrAM5CGqDHUbCGl8AeLyglO9VtCPKfqgXLqZKW1N2ruQ9ULh1J7pIEPtoDWqYwgTQBsWLymwP8kGTz3DlNwSLRT6her8VUp3DOkuDWtDqI1cyOgmr+Mg5KnXJvClxHP2B51fIdixM/TIMQ/My5dDQXxGq9JCJB2hK+I9Nh8BBvCAvx7OUJJ7SpfjX3kVNhoZUdfKomUUvz3UKr3YiIBktaKcol4PzUiaU7FNGx35NgrJ4lKJ3LPdDuUhxtGR5P0TfWFDYAKm5IYelCbOuaaMI50SmWIUCesgihyvay4rFdkWgYG1In7NPKR7CTF+0fcVUWEfNPWF4ZEQdjopULIVGYRfmCiwX1YXSJBYedTeSybF2X27WCAi6GHRVp+VkBhmiXYs1OoeJ/2RgjJ9CX8OohWHnKNCqvTNwykkWHjK8QC1Is3kRTFVPQ7e2rPuDPf5xTrAtqbuDH8c1olEaRs6ThjQN7WmElkjcuZE80ssy91d99+YazYKp0rN8V90A6Q1s3rf2GVzjqjCqXUMTpZoSxoPASOTaq3Hep8a5qRLW+/0Mgq/nVS3HaEpTuk07wsZCPtXxEDbyggRV5ITOdrNUTInHRHnMqc08OjLl8zmNFcK5ecBdIls1Ty8r7emZnlbp+TUG+UblWgqSBBbDaMjsqaxRLw6Mfh57nletigguYdPkWf3Py31xnHfmeXtN7sae3UIXBEA9/Npy+rPZ+TchpAsfutIY5v1N37u7p3tzqjjz9AvIX483dC3FDQBq8hlZhTJ2q4eiVUqZOTFBPzYQzC5aM8Q84r36zkVajcomogC5AkF8EOxpGtHCixAHczbRffy1CCvxRtjEaHYczP2QaJilp+kDbviFyqOBTTaiSoC2HLr+UV3B7WCbWKOzqFu+ANOtYWB6zi6HlXjXcrXQosLoyQYGDdQoh5VFoUTdHbzeJE0C8Pp2FEU/BlWP+s8XjwN9cW5vW3+jIcsmauAUs7Mc25anItge/sysVrdmV4T+kexWk7uJJGwWbzSfgrnLTqaCnLj/hzqset9QdWFY7rpyQzej8ni7Z7rQB+jXtnq1b5r+zS96QT+/Lg2+qAv8eDnzjeuRnAb+t+y5w8w/mvUVcESmfEbnEK7ofyFNyDaeeyw+XeNRoLelfAU9fnoObkt7GD/F5calu/bdFPb9ptrt93RRlFTjeElhTyqvOMYpZiCrw/uyjscTEBcj90S+PGoYbrcJUq3oLN4g42+p/W9sfIg9LUod9eqCJWXQ1WieHq3BCA7w17ZFJQl0m7k4DycOiU5TjH2g4FRzbk5Q7V3461w6Pjg5BwQ8xg7VcYVILGwm2iRoBCg/zNtESDxGHRceZkSyLpn6Ccp8vna0g2i1WyK24UZhi8kw7NNaYxAH17xvtw8YX9C9e7xQDmp8ZcIc7lXX0hOyKkj7cV/A7zJVAJ1bd4h6UV1e6U1S/nboRcXEE2T/i0V/aj1BV3v70I7mKVgEYQQJuAcvN7uqNSPn8GFHvJU1pKb+BaT5Rr07YX5mbYtbuhoRR2Hp2+uLVK5ErlRpAyRTGBuTwp5Bdx9AVuJz3K9FzQQQU2pNDMmFTdHS+8AdoTlRdfBPn6EQlEjEHRTEDQG8IW7Y8yufKulVeqY332ro7rft241qa8riNXS7kH9MwA4BO7oUz5y+P/Q5M7C64g6d9BxkKuqQ1il6/oCM7Aohl28D902jJNNNpm9YuFdsxvAsj1HDLaQ/sOw63nfawl8/fc2yrfUcKbRiNic67jQLbGI3ePPtYGSySmJmPP1XZHZD1KgH9maFKZtl/tO8uCDTkUdISMVElO1ElQGv+/OpMpJJyalOakCm7Aje0pO4coi/X5VckKf41y9VI6K244rlEqSccCCcsT8dI/dhVrgHDCVmyVanJX6siTXvgDm1rahjt4WTQaU92Fmk2A7fqM5smVPgBRFtM6ODT6uSY8YcYLRgOLDWfPMRzgPEPtNShAJBPz5xnL/S6bso0X8jk2eRG1leUAY5U6Vnc7t2qNDfNneTkCRv4X7jurNEA6dzk6UFxq7VRSDVyxkKj/nsIrIWLXB9EH1zTnBbX8wSPLM0wuyErE4OJcDE7yYlrQZgrLVcFW7KYsIFgpbTdTmrqdrI6YBr1jKn7dK32e5EMwcEZaurAAUpgtsXaQP45Cy0AaNuovF1nb2OJ7fTdlBW2y79kqf20RR4CP5cD2/H1Sr7+3c1iY7uba3f8hd3Xn+2e86j5hTza3bHKvD2TZlzdT2xzMWn/nNn1pN08kQIiO7f4+SXn38U16r9Q3Pm5kTyQYPGyzpWKOH0EgR/sq+RMKw3KnXp2r9efdA3DG1i0123XutPq0JJDrTaiSx12+ggh8JF9kVa1TkdhPiERUW8CKKC+axQ4oE4i25yqCqL+m1EEIoCcSqPV0chYhVcJjfOTxfbXW4Ue/wtn9QMiVT8AAA=="""
WAVE7_PATCH_SHA256 = "91b122efcfbf838db018d468fe9d9288fe395f9d41370fb4b725f92b9d1e78b5"
WAVE7_PATCH_GZIP_B64 = "..."

NOTEBOOK_BUILD = "wave7-w8pc-fullk-v3"
HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
HF_EXPECTED_BYTES = 491400032
HF_EXPECTED_SHA256 = "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db"
MODEL_PATH = ""

TARGET_PREFILL_TPS = 15_000.0
T4_INT8_TMAC_S = 65.0
FORCE_Q8_FOR_PTX = True
MICRO_REPEATS = 2
PRODUCTION_REPEATS = 2
COLD_ITERS = 3
WARMUP_ITERS = 3
MEASURE_ITERS = 10
RUN_NCU = True

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-ceiling-wave7-{RUN_ID}"
META_REPO = ROOT / "meta"
BASE_DIR = ROOT / "wave4"
CAND_DIR = ROOT / "wave7"
RESULTS = ROOT / "results"
BASE_TARGET = ROOT / "target-wave4"
CAND_TARGET = ROOT / "target-wave7"
for path in (ROOT, RESULTS):
    path.mkdir(parents=True, exist_ok=False)

def run(cmd, cwd=None, env=None, timeout=7200, check=True):
    merged = dict(os.environ)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
        stdin=subprocess.DEVNULL,
    )
    if check and proc.returncode:
        tail = (proc.stdout + "\n" + proc.stderr)[-5000:]
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(map(str, cmd))}\n{tail}")
    return proc

def save_log(name, proc):
    path = RESULTS / name
    path.write_text(
        f"$ {' '.join(map(str, proc.args))}\nexit={proc.returncode}\n\n"
        f"--- stdout ---\n{proc.stdout}\n--- stderr ---\n{proc.stderr}",
        encoding="utf-8",
    )
    return path

gpu_proc = run([
    "nvidia-smi",
    "--query-gpu=index,name,compute_cap,memory.total,driver_version",
    "--format=csv,noheader,nounits",
], timeout=60)
gpu_rows = [line.strip() for line in gpu_proc.stdout.splitlines() if line.strip()]
if not gpu_rows:
    raise SystemExit("No NVIDIA GPU is visible. Enable a Kaggle GPU accelerator.")
print("Visible GPUs:")
for row in gpu_rows:
    print(" ", row)
first = [x.strip() for x in gpu_rows[0].split(",")]
if len(first) < 5 or "T4" not in first[1] or first[2] != "7.5":
    raise SystemExit(f"GPU 0 must be NVIDIA T4 compute capability 7.5; got: {gpu_rows[0]}")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if shutil.which("cargo") is None:
    installer = run([
        "bash", "-lc",
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
        "sh -s -- -y --profile minimal",
    ], timeout=1200)
    save_log("rustup-install.log", installer)
os.environ["PATH"] = str(Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
if shutil.which("cargo") is None:
    raise SystemExit("Rust installation did not expose cargo.")

clone = run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], timeout=1800)
save_log("git-clone.log", clone)
have_rev = run(["git", "cat-file", "-e", f"{BASE_REV}^{{commit}}"], cwd=META_REPO, check=False)
if have_rev.returncode:
    fetch = run(["git", "fetch", "--depth", "1", "origin", BASE_REV], cwd=META_REPO, timeout=1800)
    save_log("git-fetch-base.log", fetch)

run(["git", "worktree", "add", "--detach", BASE_DIR, BASE_REV], cwd=META_REPO)
run(["git", "worktree", "add", "--detach", CAND_DIR, BASE_REV], cwd=META_REPO)
actual_base = run(["git", "rev-parse", "HEAD"], cwd=BASE_DIR).stdout.strip()
if actual_base != BASE_REV:
    raise SystemExit(f"Checkout mismatch: expected {BASE_REV}, got {actual_base}")

def decode_patch(payload, expected_sha, name):
    raw = gzip.decompress(base64.b64decode(payload, validate=True))
    digest = hashlib.sha256(raw).hexdigest()
    if digest != expected_sha:
        raise SystemExit(f"Embedded {name} patch digest mismatch: {digest}")
    path = RESULTS / f"{name}.patch"
    path.write_bytes(raw)
    return path

wave3_patch = decode_patch(WAVE3_PATCH_GZIP_B64, WAVE3_PATCH_SHA256, "wave3")
wave4_patch = decode_patch(WAVE4_PATCH_GZIP_B64, WAVE4_PATCH_SHA256, "wave4")
wave7_patch = decode_patch(WAVE7_PATCH_GZIP_B64, WAVE7_PATCH_SHA256, "wave7")
for tree in (BASE_DIR, CAND_DIR):
    for patch in (wave3_patch, wave4_patch):
        run(["git", "apply", "--check", patch], cwd=tree)
        run(["git", "apply", "--whitespace=nowarn", patch], cwd=tree)
run(["git", "apply", "--check", wave7_patch], cwd=CAND_DIR)
run(["git", "apply", "--whitespace=nowarn", wave7_patch], cwd=CAND_DIR)
run(["git", "apply", "--check", "--reverse", wave7_patch], cwd=CAND_DIR)

base_status = run(["git", "status", "--porcelain"], cwd=BASE_DIR).stdout.strip()
cand_status = run(["git", "status", "--porcelain"], cwd=CAND_DIR).stdout.strip()
if not base_status or not cand_status:
    raise SystemExit("Wave 4/Wave 7 source trees were not patched as expected.")

base_main_text = (BASE_DIR / "glcuda/src/kernels/glcuda.ptx").read_text(encoding="ascii")
cand_main_text = (CAND_DIR / "glcuda/src/kernels/glcuda.ptx").read_text(encoding="ascii")
base_sm75 = (BASE_DIR / "glcuda/src/kernels/glcuda_sm75.ptx").read_text(encoding="ascii")
cand_sm75 = (CAND_DIR / "glcuda/src/kernels/glcuda_sm75.ptx").read_text(encoding="ascii")
base_mod = (BASE_DIR / "glcuda/src/kernels/mod.rs").read_text(encoding="utf-8")
cand_mod = (CAND_DIR / "glcuda/src/kernels/mod.rs").read_text(encoding="utf-8")
cand_loader = (CAND_DIR / "glcuda/src/loader.rs").read_text(encoding="utf-8")
cand_cache = (CAND_DIR / "glcuda/src/cache.rs").read_text(encoding="utf-8")
cand_runner = (CAND_DIR / "glcuda/src/runner.rs").read_text(encoding="utf-8")

def kernel_body(text, entry):
    start = text.index(entry)
    end = text.index("\n}\n", start) + 2
    return text[start:end]

grid64_entry = ".visible .entry gl_gemm_mma_q8("
r256_entry = ".visible .entry gl_gemm_mma_q8_r256("
w8pc_entry = ".visible .entry gl_gemm_mma_w8pc("
attn_entry = ".visible .entry gl_attn_decode_rows_f32("
w8pc_body = kernel_body(cand_sm75, w8pc_entry)
markers = {
    "retained_attention_unchanged": kernel_body(base_main_text, attn_entry) == kernel_body(cand_main_text, attn_entry),
    "retained_grid64_unchanged": kernel_body(base_sm75, grid64_entry) == kernel_body(cand_sm75, grid64_entry),
    "retained_r256_unchanged": kernel_body(base_sm75, r256_entry) == kernel_body(cand_sm75, r256_entry),
    "wave4_has_w8pc": base_sm75.count(w8pc_entry) + base_main_text.count("gl_quantize_q8_rows"),
    "wave7_w8pc_entries": cand_sm75.count(w8pc_entry),
    "wave7_row_quantizers": cand_main_text.count(".visible .entry gl_quantize_q8_rows("),
    "wave7_quantizer_declared_smem": cand_main_text.count(".shared .align 4 .b8 sm_w8q[36];"),
    "wave7_w8pc_gemv": cand_main_text.count(".visible .entry gl_gemv_w8pc("),
    "wave7_w8pc_mma": w8pc_body.count("mma.sync.aligned.m8n8k16.row.col.s32.s8.s8.s32"),
    "wave7_w8pc_vector_stores": w8pc_body.count("st.global.v2.f32"),
    "wave7_w8pc_a_stages": w8pc_body.count("st.shared.u64"),
    "wave7_w8pc_grid_y": w8pc_body.count("%ctaid.y"),
    "wave7_w8pc_smem_a": w8pc_body.count("sm_w8a[3072]"),
    "wave7_w8pc_smem_scales": w8pc_body.count("sm_w8xs[256]"),
    "wave7_fullk_converts": w8pc_body.count("cvt.rn.f32.s32"),
    "wave7_barrier_sites": w8pc_body.count("bar.sync 0"),
    "wave7_loader_banner": cand_loader.count("GLCUDA_W8PC: per-output weight scales"),
    "wave7_cache_suffix": cand_cache.count('".w8pc"'),
    "wave7_dispatch": cand_runner.count("k.gemm_mma_w8pc("),
    "rejected_r128_absent": sum(x.count("gl_gemm_mma_q8_r128") + x.count("GLCUDA_R128") for x in (cand_sm75, cand_mod, cand_runner)),
    "cp_async_instructions": sum(line.lstrip().startswith("cp.async") for line in cand_sm75.splitlines()),
    "worst_case_s32": 4864 * 127 * 127,
}
expected = {
    "retained_attention_unchanged": True,
    "retained_grid64_unchanged": True,
    "retained_r256_unchanged": True,
    "wave4_has_w8pc": 0,
    "wave7_w8pc_entries": 1,
    "wave7_row_quantizers": 1,
    "wave7_quantizer_declared_smem": 1,
    "wave7_w8pc_gemv": 1,
    "wave7_w8pc_mma": 16,
    "wave7_w8pc_vector_stores": 8,
    "wave7_w8pc_a_stages": 1,
    "wave7_w8pc_grid_y": 1,
    "wave7_w8pc_smem_a": 1,
    "wave7_w8pc_smem_scales": 1,
    "wave7_fullk_converts": 16,
    "wave7_barrier_sites": 2,
    "wave7_loader_banner": 1,
    "wave7_cache_suffix": 1,
    "wave7_dispatch": 1,
    "rejected_r128_absent": 0,
    "cp_async_instructions": 0,
    "worst_case_s32": 78_451_456,
}
if markers != expected:
    raise SystemExit(f"Wave 7 structural markers changed:\nexpected={expected}\nactual={markers}")
for name, text in (("main", cand_main_text), ("sm75", cand_sm75)):
    if "\r" in text or not text.isascii():
        raise SystemExit(f"Candidate {name} PTX must remain ASCII with LF endings.")

print(f"Notebook  {NOTEBOOK_BUILD}")
print(f"Baseline  Wave 3 {WAVE3_PATCH_SHA256} + Wave 4 {WAVE4_PATCH_SHA256}")
print(f"Candidate Wave 7 patch {WAVE7_PATCH_SHA256}")
print(f"Run root  {ROOT}")
print("Structural markers:", markers)


## 2 - Fetch the pinned production model

Internet must be enabled. The resumable download is accepted only after byte-size, GGUF magic, and SHA-256 checks pass.


In [ ]:
print(f"MODEL FETCH START [{NOTEBOOK_BUILD}]")
sys.stdout.flush()

import urllib.error
import urllib.request

MODEL_CACHE = WORK / "models"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
model_dest = MODEL_CACHE / HF_FILENAME
model_part = model_dest.with_name(model_dest.name + ".part")
model_url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"

def file_sha256(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def park_invalid(path, reason):
    parked = path.with_name(path.name + f".invalid-{reason}-{RUN_ID}")
    path.replace(parked)
    print(f"Parked invalid cache file: {parked}")

def validate_model(path):
    if not path.is_file():
        return False, "missing"
    size = path.stat().st_size
    if size != HF_EXPECTED_BYTES:
        return False, f"size-{size}"
    with path.open("rb") as handle:
        if handle.read(4) != b"GGUF":
            return False, "magic"
    digest = file_sha256(path)
    if digest != HF_EXPECTED_SHA256:
        return False, f"sha256-{digest[:12]}"
    return True, digest

valid, detail = validate_model(model_dest)
if valid:
    print(f"Reusing verified model: {model_dest}")
else:
    if model_dest.exists():
        park_invalid(model_dest, detail)
    if model_part.exists() and model_part.stat().st_size > HF_EXPECTED_BYTES:
        park_invalid(model_part, f"oversize-{model_part.stat().st_size}")
    if model_part.exists() and model_part.stat().st_size == HF_EXPECTED_BYTES:
        partial_valid, partial_detail = validate_model(model_part)
        if partial_valid:
            model_part.replace(model_dest)
        else:
            park_invalid(model_part, partial_detail)
    if not model_dest.exists():
        start = model_part.stat().st_size if model_part.exists() else 0
        headers = {"User-Agent": "GwenLand-Wave6-Kaggle/1.0"}
        if start:
            headers["Range"] = f"bytes={start}-"
            print(f"Resuming model download at {start / 2**20:.1f} MiB")
        else:
            print(f"Downloading {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}")
        request = urllib.request.Request(model_url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
        except urllib.error.HTTPError as exc:
            raise SystemExit(f"Model download HTTP {exc.code}: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        except urllib.error.URLError as exc:
            raise SystemExit(f"Model download connection failed: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        status = getattr(response, "status", response.getcode())
        if start and status != 206:
            print(f"Server ignored Range (HTTP {status}); restarting the partial download.")
            start = 0
        mode = "ab" if start and status == 206 else "wb"
        downloaded = start
        last_print = time.monotonic()
        with response, model_part.open(mode) as output:
            while True:
                block = response.read(8 * 1024 * 1024)
                if not block:
                    break
                output.write(block)
                downloaded += len(block)
                now = time.monotonic()
                if now - last_print >= 5:
                    pct = 100.0 * downloaded / HF_EXPECTED_BYTES
                    print(f"  {downloaded / 2**20:.1f} / {HF_EXPECTED_BYTES / 2**20:.1f} MiB ({pct:.1f}%)")
                    last_print = now
        part_valid, part_detail = validate_model(model_part)
        if not part_valid:
            park_invalid(model_part, part_detail)
            raise SystemExit(f"Downloaded GGUF failed integrity validation: {part_detail}")
        model_part.replace(model_dest)
valid, digest = validate_model(model_dest)
if not valid:
    raise SystemExit(f"Final GGUF validation failed: {digest}")
MODEL_PATH = str(model_dest)
MODEL_FETCH = {
    "repo": HF_REPO,
    "revision": HF_REVISION,
    "filename": HF_FILENAME,
    "url": model_url,
    "bytes": model_dest.stat().st_size,
    "sha256": digest,
    "path": MODEL_PATH,
}
(RESULTS / "model-fetch.json").write_text(json.dumps(MODEL_FETCH, indent=2), encoding="utf-8")
print(json.dumps(MODEL_FETCH, indent=2))


## 3 - PTX assembly and occupancy gate

Both PTX modules are assembled for sm_75. The W8PC MMA must use 3,328 B shared memory, at most 85 registers, and zero spills; with a 256-thread CTA that preserves at least three CTAs or 24 resident warps per T4 SM. The row quantizer and dp4a fallback must also be spill-free.


In [ ]:
ptxas = shutil.which("ptxas")
if ptxas is None:
    candidates = sorted(Path("/usr/local").glob("cuda*/bin/ptxas"), reverse=True)
    ptxas = str(candidates[0]) if candidates else None
if ptxas is None:
    raise SystemExit("ptxas is required for Wave 7 but was not found in the Kaggle image.")

tool_versions = {}
for name, cmd in {
    "nvidia_smi": ["nvidia-smi"],
    "rustc": ["rustc", "--version", "--verbose"],
    "cargo": ["cargo", "--version"],
    "ptxas": [ptxas, "--version"],
}.items():
    p = run(cmd, check=False, timeout=120)
    tool_versions[name] = (p.stdout + p.stderr).strip()
    print(f"--- {name} ---\n{tool_versions[name][:1500]}")

def parse_ptxas_function_resources(text, fn):
    header = re.search(
        rf"(?m)^ptxas info\s*: Function properties for {re.escape(fn)}\s*$",
        text,
    )
    if not header:
        raise ValueError(f"missing Function properties header for {fn}")
    tail = text[header.end():]
    next_function = re.search(r"(?m)^ptxas info\s*: Compiling entry function\b", tail)
    block = tail[:next_function.start()] if next_function else tail
    registers = re.search(r"Used\s+(\d+)\s+registers", block)
    spills = [int(x) for x in re.findall(r"(\d+) bytes spill (?:stores|loads)", block)]
    smem = re.search(r"(\d+)\s+bytes smem", block)
    if not registers or len(spills) != 2:
        raise ValueError(f"missing register/spill counters for {fn}")
    return {
        "registers": int(registers.group(1)),
        "smem_bytes": int(smem.group(1)) if smem else 0,
        "spill_bytes": spills,
    }

_PTXAS_PARSE_FIXTURE = """ptxas info    : Compiling entry function 'gl_attn_decode_rows_f32'
ptxas info    : Function properties for gl_attn_decode_rows_f32
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 40 registers, used 1 barriers, 388 bytes cmem[0]
ptxas info    : Compiling entry function 'gl_attn_decode_rows_f32_longer'
ptxas info    : Function properties for gl_attn_decode_rows_f32_longer
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 42 registers, used 1 barriers, 16420 bytes smem, 388 bytes cmem[0]
"""
assert parse_ptxas_function_resources(_PTXAS_PARSE_FIXTURE, "gl_attn_decode_rows_f32") == {
    "registers": 40, "smem_bytes": 0, "spill_bytes": [0, 0]
}
assert parse_ptxas_function_resources(_PTXAS_PARSE_FIXTURE, "gl_attn_decode_rows_f32_longer") == {
    "registers": 42, "smem_bytes": 16420, "spill_bytes": [0, 0]
}

def assemble_module(label, src, module_name, functions):
    cubin = RESULTS / f"{label}-{Path(module_name).stem}.cubin"
    ptx_file = src / "glcuda/src/kernels" / module_name
    p = run([ptxas, "-arch=sm_75", "-v", ptx_file, "-o", cubin], cwd=src, timeout=600, check=False)
    save_log(f"ptxas-{label}-{Path(module_name).stem}.log", p)
    text = p.stdout + "\n" + p.stderr
    print(f"\n--- ptxas {label}/{module_name} ---\n{text}")
    if p.returncode:
        raise SystemExit(f"ptxas failed for {label}/{module_name}")
    resources = {}
    for fn in functions:
        try:
            resources[fn] = parse_ptxas_function_resources(text, fn)
        except ValueError as exc:
            raise SystemExit(f"Could not parse {label}/{fn}: {exc}") from exc
        if any(resources[fn]["spill_bytes"]):
            raise SystemExit(f"PTXAS spill gate failed for {label}/{fn}: {resources[fn]}")
    return {"cubin": str(cubin), "resources": resources, "log": text}

PTXAS = {}
PTXAS["wave4"] = {
    "main": assemble_module("wave4", BASE_DIR, "glcuda.ptx", ("gl_attn_decode_rows_f32", "gl_attn_rows_probe")),
    "sm75": assemble_module("wave4", BASE_DIR, "glcuda_sm75.ptx", ("gl_gemm_mma_q8", "gl_gemm_mma_q8_r256")),
}
PTXAS["wave7"] = {
    "main": assemble_module("wave7", CAND_DIR, "glcuda.ptx", ("gl_attn_decode_rows_f32", "gl_attn_rows_probe", "gl_quantize_q8_rows", "gl_gemv_w8pc")),
    "sm75": assemble_module("wave7", CAND_DIR, "glcuda_sm75.ptx", ("gl_gemm_mma_q8", "gl_gemm_mma_q8_r256", "gl_gemm_mma_w8pc")),
}

for fn in ("gl_attn_decode_rows_f32", "gl_attn_rows_probe"):
    if PTXAS["wave4"]["main"]["resources"][fn] != PTXAS["wave7"]["main"]["resources"][fn]:
        raise SystemExit(f"Wave 7 unexpectedly changed retained main-PTX resources for {fn}.")
for fn in ("gl_gemm_mma_q8", "gl_gemm_mma_q8_r256"):
    if PTXAS["wave4"]["sm75"]["resources"][fn] != PTXAS["wave7"]["sm75"]["resources"][fn]:
        raise SystemExit(f"Wave 7 unexpectedly changed retained sm75 resources for {fn}.")

w8_res = PTXAS["wave7"]["sm75"]["resources"]["gl_gemm_mma_w8pc"]
q_res = PTXAS["wave7"]["main"]["resources"]["gl_quantize_q8_rows"]
gemv_res = PTXAS["wave7"]["main"]["resources"]["gl_gemv_w8pc"]
if w8_res["smem_bytes"] != 3_328:
    raise SystemExit(f"Wave 7 W8PC shared-memory gate failed: {w8_res}")
if w8_res["registers"] > 85:
    raise SystemExit(f"Wave 7 W8PC register gate failed (>85 loses the 3-CTA tier): {w8_res}")
# The PTX declaration is 36 B (checked structurally above). On sm_75,
# ptxas rounds this allocation to a 16-byte boundary and reports 48 B.
if q_res["smem_bytes"] != 48:
    raise SystemExit(f"Wave 7 row-quantizer allocated shared-memory gate failed: {q_res}")
cta_by_regs = 65_536 // (w8_res["registers"] * 256)
cta_by_smem = 65_536 // w8_res["smem_bytes"]
cta_by_threads = 1_024 // 256
resident_ctas = min(16, cta_by_regs, cta_by_smem, cta_by_threads)
resident_warps = resident_ctas * 8
if resident_ctas < 3 or resident_warps < 24:
    raise SystemExit(f"Wave 7 occupancy gate failed: CTAs={resident_ctas}, warps={resident_warps}, resources={w8_res}")

W8PC_RESOURCES = {
    "grid64": PTXAS["wave4"]["sm75"]["resources"]["gl_gemm_mma_q8"],
    "w8pc": w8_res,
    "r256": PTXAS["wave4"]["sm75"]["resources"]["gl_gemm_mma_q8_r256"],
    "row_quantizer": q_res,
    "w8pc_gemv": gemv_res,
    "resident_ctas": resident_ctas,
    "resident_warps": resident_warps,
}
PTXAS_OK = True
print("\nPTXAS resource gate PASS")
print(json.dumps(W8PC_RESOURCES, indent=2))


## 4 - Hardware and model correctness gates

Both arms run lib, parity, forward, and graph-replay suites on the T4; CUDA skips are fatal. Candidate parity explicitly covers the row quantizer, W8PC dp4a GEMV, ragged cross-CTA MMA tails, dim-896, and the 4864-wide down projection. Production sessions additionally require exact greedy next-token agreement with the glproc oracle.


In [ ]:
def cargo_env(target):
    return {
        "CARGO_TARGET_DIR": str(target),
        "RUST_BACKTRACE": "1",
        "CUDA_VISIBLE_DEVICES": "0",
    }

def cargo_run(label, src, target, args, log_name, timeout=7200):
    p = run(["cargo", *args], cwd=src, env=cargo_env(target), timeout=timeout, check=False)
    save_log(log_name, p)
    hay = p.stdout + "\n" + p.stderr
    if p.returncode:
        raise SystemExit(f"{label} failed (exit {p.returncode}); see {RESULTS / log_name}\n{hay[-3000:]}")
    return hay

TEST_RESULTS = {}
for label, src, target in [
    ("wave4", BASE_DIR, BASE_TARGET),
    ("wave7", CAND_DIR, CAND_TARGET),
]:
    print(f"\n=== {label}: host library tests ===")
    host = cargo_run(
        label, src, target,
        ["test", "--locked", "-p", "glcuda", "--release", "--lib", "--", "--nocapture"],
        f"test-{label}-lib.log",
    )
    TEST_RESULTS[f"{label}_lib"] = host
    for suite in ("parity", "forward", "graph_replay"):
        print(f"=== {label}: {suite} ===")
        hay = cargo_run(
            f"{label}/{suite}", src, target,
            ["test", "--locked", "-p", "glcuda", "--release", "--test", suite,
             "--", "--test-threads=1", "--nocapture"],
            f"test-{label}-{suite}.log",
        )
        if "SKIP: no CUDA driver/device" in hay:
            raise SystemExit(f"{label}/{suite} silently skipped CUDA device tests.")
        matches = re.findall(r"test result: (ok|FAILED)\. (\d+) passed; (\d+) failed", hay)
        if not matches or any(state != "ok" or int(failed) != 0 for state, _, failed in matches):
            raise SystemExit(f"Could not prove {label}/{suite} passed on hardware.")
        TEST_RESULTS[f"{label}_{suite}"] = {"summaries": matches, "skip_count": 0}
        print(" ", matches[-1])
        if label == "wave7" and suite == "parity":
            for test_name in ("gemv_w8pc_matches_row_scaled_reference", "gemm_mma_w8pc_matches_row_scaled_reference"):
                if not re.search(rf"test {test_name} \.\.\. ok", hay):
                    raise SystemExit(f"Candidate hardware test did not execute successfully: {test_name}")

for label, src, target in [
    ("wave4", BASE_DIR, BASE_TARGET),
    ("wave7", CAND_DIR, CAND_TARGET),
]:
    cargo_run(
        f"{label}/bench build", src, target,
        ["build", "--locked", "--release", "-p", "glcuda", "--example", "bench"],
        f"build-{label}-bench.log",
    )
    cargo_run(
        f"{label}/glbench build", src, target,
        ["build", "--locked", "--release", "-p", "glbench"],
        f"build-{label}-glbench.log",
    )

BINS = {
    "wave4": {"bench": BASE_TARGET / "release/examples/bench", "glbench": BASE_TARGET / "release/glbench"},
    "wave7": {"bench": CAND_TARGET / "release/examples/bench", "glbench": CAND_TARGET / "release/glbench"},
}
for arm, bins in BINS.items():
    for kind, path in bins.items():
        if not path.exists():
            raise SystemExit(f"Missing {arm} {kind} binary: {path}")

CORRECTNESS_OK = True
print("\nCorrectness gate passed for Wave 4 and Wave 7 on the real T4.")


## 5 - Diagnostic work-reduction budget

This is a contract-level diagnostic, not a retention gate. For Qwen2.5-0.5B at 244 tokens it counts activation-quantizer CTAs and K32 scale folds removed by the row-scale contract. Production glbench remains the decision.


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")

prompt_tokens = 244
layers = 24
# Per layer: qkv/wo/gate-up use K=896 (28 K32 blocks); down uses K=4864
# (152 blocks). Four activation quantizer calls per layer, plus the LM head.
old_quantizer_ctas = layers * prompt_tokens * (28 + 28 + 28 + 152) + 28
new_quantizer_ctas = layers * prompt_tokens * 4 + 1
old_scale_folds = layers * prompt_tokens * (
    1152 * 28 + 896 * 28 + 9728 * 28 + 896 * 152
) + 151936 * 28
new_scale_folds = layers * prompt_tokens * (1152 + 896 + 9728 + 896) + 151936
DIAGNOSTIC_BUDGET = {
    "prompt_tokens": prompt_tokens,
    "old_quantizer_ctas": old_quantizer_ctas,
    "new_quantizer_ctas": new_quantizer_ctas,
    "quantizer_cta_reduction": old_quantizer_ctas / new_quantizer_ctas,
    "old_scale_folds": old_scale_folds,
    "new_scale_folds": new_scale_folds,
    "scale_fold_reduction": old_scale_folds / new_scale_folds,
    "worst_case_s32": 4864 * 127 * 127,
    "int32_headroom": (2**31 - 1) / (4864 * 127 * 127),
}
print(json.dumps(DIAGNOSTIC_BUDGET, indent=2))


## 6 - Production glbench A/B

Two order-reversed sessions, each with 3 warmups and 10 measured iterations. Every session must also obtain exact greedy next-token agreement from the glproc oracle. Retention requires at least +5% P50 and mean in both pairs, no >5% P95/decode regression, green correctness, zero spills, and the expected occupancy tier.


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")

model = Path(MODEL_PATH)
if not model.is_file() or model.stat().st_size < 10_000_000:
    raise SystemExit(f"Model path is not a plausible GGUF: {model}")

prompt_unit = (
    "Measure this deterministic systems prompt carefully. Explain how token-parallel "
    "integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
)
FIXED_PROMPT = prompt_unit * 8
arms = [
    ("wave4_attn_dsmem", "wave4", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1"}),
    ("wave7_w8pc", "wave7", {"GLCUDA_W8PC": "1", "GLCUDA_GRID2D": "1"}),
]
arm_map = {label: (build, env) for label, build, env in arms}

def percentile(values, q):
    values = sorted(values)
    index = (len(values) - 1) * q
    lo, hi = math.floor(index), math.ceil(index)
    if lo == hi:
        return values[lo]
    return values[lo] * (hi - index) + values[hi] * (index - lo)

def session_stats(path, expected_iters=MEASURE_ITERS, require_exact_oracle=False):
    data = json.loads(path.read_text(encoding="utf-8"))
    engine_blob = json.dumps(data.get("engine", {}), sort_keys=True).lower()
    if "glcuda" not in engine_blob:
        raise RuntimeError(f"Session did not record glcuda engine: {data.get('engine')}")
    validation = data.get("validation") or {}
    findings = validation.get("findings", [])
    parity = [f for f in findings if f.get("check") == "parity"]
    other_errors = [f for f in findings if f.get("severity") == "error" and f.get("check") != "parity"]
    if other_errors:
        raise RuntimeError(f"Session failed non-parity validation: {other_errors}")
    if not parity:
        raise RuntimeError("Session did not record the requested glproc oracle check.")
    match = re.search(r"(\d+)/(\d+) tokens match oracle", parity[-1].get("message", ""))
    if not match:
        raise RuntimeError(f"Could not parse glproc oracle evidence: {parity[-1]}")
    oracle_prefix, oracle_compared = map(int, match.groups())
    oracle_exact = oracle_compared > 0 and oracle_prefix == oracle_compared
    if require_exact_oracle and not oracle_exact:
        raise RuntimeError(f"Retained baseline lost exact glproc parity: {parity[-1]}")
    iterations = data.get("measurements", {}).get("iterations", [])
    prefill_ms = [float(x.get("prefill_ms", 0.0)) for x in iterations]
    decode_ms = [float(x.get("decode_ms", 0.0)) for x in iterations]
    prompt_counts = [int(x.get("prompt_tokens", 0)) for x in iterations]
    if len(iterations) != expected_iters or any(x <= 0 for x in prefill_ms + decode_ms):
        raise RuntimeError(f"Expected {expected_iters} valid iterations, got {iterations}")
    if len(set(prompt_counts)) != 1 or prompt_counts[0] <= 0:
        raise RuntimeError(f"Prompt token count changed: {prompt_counts}")
    prefill_tps = [prompt_counts[0] * 1000.0 / x for x in prefill_ms]
    decode_tps = [1000.0 / x for x in decode_ms]
    return {
        "prompt_tokens": prompt_counts[0],
        "prefill_tps_samples": prefill_tps,
        "prefill_mean": statistics.mean(prefill_tps),
        "prefill_p50": percentile(prefill_tps, 0.50),
        "prefill_p10": percentile(prefill_tps, 0.10),
        "prefill_latency_p50_ms": percentile(prefill_ms, 0.50),
        "prefill_latency_p95_ms": percentile(prefill_ms, 0.95),
        "prefill_latency_p99_ms": percentile(prefill_ms, 0.99),
        "decode_p50": percentile(decode_tps, 0.50),
        "validation_passed": bool(validation.get("passed", False)),
        "oracle_matching_prefix": oracle_prefix,
        "oracle_compared": oracle_compared,
        "oracle_exact": oracle_exact,
    }

PROD_RECORDS = []
for repeat in range(PRODUCTION_REPEATS):
    rotated = arms[repeat % len(arms):] + arms[:repeat % len(arms)]
    for label, build_arm, extra_env in rotated:
        archive = RESULTS / f"glbench-{repeat}-{label}.json"
        cmd = [
            BINS[build_arm]["glbench"], "run", "--engine", "glcuda",
            "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1",
            "--cold-iters", str(COLD_ITERS), "--warmup", str(WARMUP_ITERS),
            "--iters", str(MEASURE_ITERS), "--temperature", "0", "--seed", "42",
            "--kind", "prefill", "--verify-against", "glproc", "--out", archive,
        ]
        src = BASE_DIR if build_arm == "wave4" else CAND_DIR
        p = run(cmd, cwd=src, env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
        save_log(f"glbench-{repeat}-{label}.log", p)
        hay = p.stdout + "\n" + p.stderr
        if p.returncode:
            raise SystemExit(f"glbench {label} repeat {repeat} failed.")
        expected_banners = ["2-D token-grid prefill GEMM enabled", "dynamic-shared prefill attention enabled"]
        if build_arm == "wave4":
            expected_banners += ["GLCUDA_FORCE_Q8:"]
            forbidden = ["GLCUDA_W8PC:", "W8PC MMA enabled", "r128 prefill GEMM enabled"]
        else:
            expected_banners += ["GLCUDA_W8PC: per-output weight scales", "W8PC MMA enabled"]
            forbidden = ["GLCUDA_FORCE_Q8:", "r128 prefill GEMM enabled"]
        for banner in expected_banners:
            if banner not in hay:
                raise SystemExit(f"{label} did not confirm required dispatch banner: {banner}")
        for banner in forbidden:
            if banner in hay:
                raise SystemExit(f"{label} unexpectedly enabled: {banner}")
        if "r256 prefill GEMM enabled" in hay:
            raise SystemExit(f"{label} accidentally enabled r256.")
        stats = session_stats(archive, require_exact_oracle=(build_arm == "wave4"))
        rec = {"repeat": repeat, "arm": label, "archive": str(archive), **stats}
        PROD_RECORDS.append(rec)
        print(
            f"{label:18s} repeat {repeat}: P50 {stats['prefill_p50']:.1f} tok/s | "
            f"mean {stats['prefill_mean']:.1f} | P95 latency {stats['prefill_latency_p95_ms']:.2f} ms | "
            f"decode P50 {stats['decode_p50']:.1f} | oracle "
            f"{stats['oracle_matching_prefix']}/{stats['oracle_compared']}"
        )

PROD_SUMMARY = []
for label, _, _ in arms:
    rows = [x for x in PROD_RECORDS if x["arm"] == label]
    PROD_SUMMARY.append({
        "arm": label,
        "session_p50_median": statistics.median(x["prefill_p50"] for x in rows),
        "session_mean_median": statistics.median(x["prefill_mean"] for x in rows),
        "session_p95_latency_median_ms": statistics.median(x["prefill_latency_p95_ms"] for x in rows),
        "decode_p50_median": statistics.median(x["decode_p50"] for x in rows),
        "sessions": len(rows),
        "oracle_prefixes": [f"{x['oracle_matching_prefix']}/{x['oracle_compared']}" for x in rows],
    })

base_rows = [x for x in PROD_RECORDS if x["arm"] == "wave4_attn_dsmem"]
cand_rows = [x for x in PROD_RECORDS if x["arm"] == "wave7_w8pc"]
paired = []
for repeat in range(PRODUCTION_REPEATS):
    a = next(x for x in base_rows if x["repeat"] == repeat)
    b = next(x for x in cand_rows if x["repeat"] == repeat)
    paired.append({
        "repeat": repeat,
        "prefill_p50_delta": b["prefill_p50"] / a["prefill_p50"] - 1.0,
        "prefill_mean_delta": b["prefill_mean"] / a["prefill_mean"] - 1.0,
        "p95_latency_delta": b["prefill_latency_p95_ms"] / a["prefill_latency_p95_ms"] - 1.0,
        "decode_p50_delta": b["decode_p50"] / a["decode_p50"] - 1.0,
    })
base_tps = statistics.median(x["prefill_p50"] for x in base_rows)
cand_tps = statistics.median(x["prefill_p50"] for x in cand_rows)
WAVE7_DECISION = {
    "baseline_arm": "wave4_attn_dsmem",
    "candidate_arm": "wave7_w8pc",
    "baseline_tps": base_tps,
    "candidate_tps": cand_tps,
    "median_delta": cand_tps / base_tps - 1.0,
    "paired": paired,
    "correctness_green": bool(CORRECTNESS_OK),
}
WAVE7_DECISION["prefill_p50_pass"] = all(x["prefill_p50_delta"] >= 0.05 for x in paired)
WAVE7_DECISION["prefill_mean_pass"] = all(x["prefill_mean_delta"] >= 0.05 for x in paired)
WAVE7_DECISION["tail_pass"] = all(x["p95_latency_delta"] <= 0.05 for x in paired)
WAVE7_DECISION["decode_pass"] = all(x["decode_p50_delta"] >= -0.05 for x in paired)
WAVE7_DECISION["oracle_parity_pass"] = all(x["oracle_exact"] for x in cand_rows)
WAVE7_DECISION["retain"] = all([
    WAVE7_DECISION["median_delta"] >= 0.05,
    WAVE7_DECISION["correctness_green"],
    WAVE7_DECISION["oracle_parity_pass"],
    WAVE7_DECISION["prefill_p50_pass"],
    WAVE7_DECISION["prefill_mean_pass"],
    WAVE7_DECISION["tail_pass"],
    WAVE7_DECISION["decode_pass"],
])
print("\nProduction summary:", json.dumps(PROD_SUMMARY, indent=2))
print("\nWave 7 decision:", json.dumps(WAVE7_DECISION, indent=2))

TELEMETRY = {}
for label, build_arm, extra_env in arms:
    archive = RESULTS / f"telemetry-{label}.json"
    cmd = [
        BINS[build_arm]["glbench"], "run", "--engine", "glcuda",
        "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1",
        "--cold-iters", "0", "--warmup", "3", "--iters", "1",
        "--temperature", "0", "--seed", "42", "--kind", "prefill", "--verify-against", "glproc", "--out", archive,
    ]
    src = BASE_DIR if build_arm == "wave4" else CAND_DIR
    p = run(cmd, cwd=src, env={"CUDA_VISIBLE_DEVICES": "0", **extra_env, "GLCUDA_TELEMETRY": "1"}, timeout=14400, check=False)
    save_log(f"telemetry-{label}.log", p)
    if p.returncode:
        raise SystemExit(f"Telemetry run failed for {label}.")
    data = json.loads(archive.read_text(encoding="utf-8"))
    stages = (((data.get("telemetry") or {}).get("prefill") or {}).get("stages") or [])
    if not stages:
        raise SystemExit(f"Telemetry produced no stages for {label}.")
    TELEMETRY[label] = {"stages": stages, "session": session_stats(archive, expected_iters=1, require_exact_oracle=(build_arm == "wave4"))}

cand_stages = TELEMETRY["wave7_w8pc"]["stages"]
stage_total_ms = sum(float(x.get("total_ms") or 0.0) for x in cand_stages)
attention_ms = sum(float(x.get("total_ms") or 0.0) for x in cand_stages if x.get("name") == "attention")
gemm_names = {"qkv", "attn_out", "ffn_gate_up", "ffn_down"}
gemm_ms = sum(float(x.get("total_ms") or 0.0) for x in cand_stages if x.get("name") in gemm_names)
prompt_tokens = cand_rows[0]["prompt_tokens"]
TARGET_ANALYSIS = {
    "prompt_tokens": prompt_tokens,
    "measured_tps": cand_tps,
    "target_tps": TARGET_PREFILL_TPS,
    "measured_prefill_ms": prompt_tokens * 1000.0 / cand_tps,
    "target_prefill_ms": prompt_tokens * 1000.0 / TARGET_PREFILL_TPS,
    "required_speedup": TARGET_PREFILL_TPS / cand_tps,
    "attention_share": attention_ms / stage_total_ms,
    "gemm_share": gemm_ms / stage_total_ms,
    "infinite_attention_ceiling_tps": cand_tps / (1.0 - attention_ms / stage_total_ms),
    "infinite_gemm_ceiling_tps": cand_tps / (1.0 - gemm_ms / stage_total_ms),
}

for repeat in range(PRODUCTION_REPEATS):
    b = RESULTS / f"glbench-{repeat}-wave4_attn_dsmem.json"
    c = RESULTS / f"glbench-{repeat}-wave7_w8pc.json"
    p = run([BINS["wave7"]["glbench"], "compare", b, c], cwd=CAND_DIR, timeout=600, check=False)
    save_log(f"compare-{repeat}-wave4-vs-wave7.log", p)
    if p.returncode:
        raise SystemExit("glbench compare failed for Wave 4 vs Wave 7.")
PROD_OK = True


## 7 - Optional Nsight Compute evidence

Profiles retained grid64 for the baseline and `gl_gemm_mma_w8pc` for the candidate when performance-counter permissions exist. `ERR_NVGPUCTRPERM` is archived but does not waive PTXAS/static gates.


In [ ]:
NCU = {"available": False, "runs": {}}
ncu = shutil.which("ncu")
if not RUN_NCU:
    print("Nsight Compute disabled by configuration.")
elif ncu is None:
    print("Nsight Compute CLI is not installed in this Kaggle image.")
else:
    listed = run([ncu, "--list-sections"], timeout=300, check=False)
    save_log("ncu-list-sections.log", listed)
    available_text = listed.stdout + "\n" + listed.stderr
    wanted = ["LaunchStats", "Occupancy", "SpeedOfLight", "WarpStateStats", "MemoryWorkloadAnalysis", "ComputeWorkloadAnalysis"]
    sections = [name for name in wanted if name in available_text]
    NCU["available"] = True
    NCU["sections"] = sections
    for label, build_arm, extra_env in arms:
        report = RESULTS / f"ncu-{label}"
        kernel_name = "regex:.*gl_gemm_mma_w8pc$" if build_arm == "wave7" else "regex:.*gl_gemm_mma_q8$"
        cmd = [ncu, "--target-processes", "all",
               "--kernel-name", kernel_name, "--launch-count", "1",
               "--force-overwrite", "--export", report]
        for section in sections:
            cmd += ["--section", section]
        if not sections:
            cmd += ["--set", "basic"]
        archive = RESULTS / f"ncu-session-{label}.json"
        cmd += [BINS[build_arm]["glbench"], "run", "--engine", "glcuda",
                "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1",
                "--cold-iters", "0", "--warmup", "0", "--iters", "1",
                "--temperature", "0", "--seed", "42", "--kind", "prefill", "--out", archive]
        src = BASE_DIR if build_arm == "wave4" else CAND_DIR
        p = run(cmd, cwd=src, env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
        save_log(f"ncu-{label}.log", p)
        text = p.stdout + "\n" + p.stderr
        permitted = p.returncode == 0 and "ERR_NVGPUCTRPERM" not in text
        NCU["runs"][label] = {"returncode": p.returncode, "permitted": permitted}
        print(f"{label:18s}: exit={p.returncode}, counters={'captured' if permitted else 'unavailable'}")
        if permitted:
            imported = run([ncu, "--import", str(report) + ".ncu-rep", "--page", "details", "--csv"], timeout=1800, check=False)
            save_log(f"ncu-{label}-details.csv", imported)


## 8 - Package the Wave 7 evidence

Produces one ZIP containing patches, PTXAS logs/cubins, hardware tests, raw glbench JSON, exact oracle findings, telemetry, optional NCU output, manifest, and the final report.


In [ ]:
ptxas_resources = {arm: {module: data["resources"] for module, data in modules.items()} for arm, modules in PTXAS.items()}
manifest = {
    "schema": "gwenland.glcuda.t4-ceiling.wave7.fetch.v1",
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "notebook_build": NOTEBOOK_BUILD,
    "gpu_rows": gpu_rows, "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "repo_url": REPO_URL, "base_rev": BASE_REV,
    "wave3_patch_sha256": WAVE3_PATCH_SHA256, "wave4_patch_sha256": WAVE4_PATCH_SHA256,
    "wave7_patch_sha256": WAVE7_PATCH_SHA256, "markers": markers, "tool_versions": tool_versions,
    "ptxas_ok": bool(globals().get("PTXAS_OK")), "ptxas_resources": ptxas_resources,
    "w8pc_resources": globals().get("W8PC_RESOURCES"), "correctness_ok": bool(globals().get("CORRECTNESS_OK")),
    "production_ok": bool(globals().get("PROD_OK")), "target_prefill_tps": TARGET_PREFILL_TPS,
    "production_repeats": PRODUCTION_REPEATS, "cold_iters": COLD_ITERS,
    "warmup_iters": WARMUP_ITERS, "measure_iters": MEASURE_ITERS,
    "model_fetch": globals().get("MODEL_FETCH"), "diagnostic_budget": globals().get("DIAGNOSTIC_BUDGET"),
    "production_summary": globals().get("PROD_SUMMARY", []), "wave7_decision": globals().get("WAVE7_DECISION"),
    "telemetry": globals().get("TELEMETRY", {}), "target_analysis": globals().get("TARGET_ANALYSIS"), "ncu": globals().get("NCU", {}),
}
(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

report = ["# glcuda T4 Ceiling - Wave 7", "", f"- notebook: {NOTEBOOK_BUILD}", f"- GPU: {gpu_rows[0]}",
    f"- baseline revision: {BASE_REV}", f"- Wave 3 patch: {WAVE3_PATCH_SHA256}", f"- retained Wave 4 patch: {WAVE4_PATCH_SHA256}",
    "- rejected Wave 5/Wave 6 patches applied: NO", f"- Wave 7 patch: {WAVE7_PATCH_SHA256}",
    f"- model: {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}", f"- model SHA-256: {manifest['model_fetch']['sha256']}",
    f"- ptxas: {'PASS' if manifest['ptxas_ok'] else 'FAIL'}", f"- hardware kernel-reference correctness: {'PASS' if manifest['correctness_ok'] else 'FAIL'}",
    f"- production glbench: {'COMPLETE' if manifest['production_ok'] else 'PENDING'}", f"- target: {TARGET_PREFILL_TPS:.0f} prefill tok/s", "",
    "## W8PC resource/contract gate", "", f"- grid64: {manifest['w8pc_resources']['grid64']}", f"- W8PC MMA: {manifest['w8pc_resources']['w8pc']}",
    f"- row quantizer: {manifest['w8pc_resources']['row_quantizer']}", f"- W8PC GEMV: {manifest['w8pc_resources']['w8pc_gemv']}",
    f"- projected resident CTAs/warps: {manifest['w8pc_resources']['resident_ctas']} / {manifest['w8pc_resources']['resident_warps']}", "",
    "## Diagnostic work reduction", "",
    f"- activation-quantizer CTAs: {manifest['diagnostic_budget']['old_quantizer_ctas']:,} -> {manifest['diagnostic_budget']['new_quantizer_ctas']:,} ({manifest['diagnostic_budget']['quantizer_cta_reduction']:.2f}x fewer)",
    f"- scale folds: {manifest['diagnostic_budget']['old_scale_folds']:,} -> {manifest['diagnostic_budget']['new_scale_folds']:,} ({manifest['diagnostic_budget']['scale_fold_reduction']:.2f}x fewer)",
    f"- worst-case s32 dot: {manifest['diagnostic_budget']['worst_case_s32']:,} ({manifest['diagnostic_budget']['int32_headroom']:.1f}x headroom)",
    "", "## Production prefill", "", "| arm | median session P50 tok/s | median session mean | median P95 latency ms | decode P50 | oracle prefixes | sessions |", "|---|---:|---:|---:|---:|---|---:|",
]
for row in manifest["production_summary"]:
    report.append(f"| {row['arm']} | {row['session_p50_median']:.1f} | {row['session_mean_median']:.1f} | {row['session_p95_latency_median_ms']:.2f} | {row['decode_p50_median']:.1f} | {', '.join(row['oracle_prefixes'])} | {row['sessions']} |")
if manifest.get("target_analysis"):
    a = manifest["target_analysis"]
    report += ["", "## 15k feasibility budget", "", f"- measured/target: {a['measured_tps']:.1f} / {a['target_tps']:.0f} tok/s", f"- required speedup: {a['required_speedup']:.2f}x", f"- measured/target prompt time: {a['measured_prefill_ms']:.2f} / {a['target_prefill_ms']:.2f} ms", f"- attention/GEMM stage share: {100*a['attention_share']:.1f}% / {100*a['gemm_share']:.1f}%", f"- infinite-attention-only ceiling: {a['infinite_attention_ceiling_tps']:.1f} tok/s", f"- infinite-GEMM-only ceiling: {a['infinite_gemm_ceiling_tps']:.1f} tok/s"]
d = manifest.get("wave7_decision")
report += ["", "## Wave 7 gate", ""]
if d:
    report += [f"- median P50 delta: {100*d['median_delta']:+.2f}%", f"- paired evidence: {json.dumps(d['paired'])}", f"- prefill P50/mean gates: {d['prefill_p50_pass']} / {d['prefill_mean_pass']}", f"- P95 tail/decode gates: {d['tail_pass']} / {d['decode_pass']}", f"- exact glproc next-token parity: {d['oracle_parity_pass']}", f"- verdict: {'RETAIN' if d['retain'] else 'REJECT/HOLD'}"]
else:
    report.append("- verdict: PENDING")
if manifest.get("ncu", {}).get("available") and not any(x.get("permitted") for x in manifest["ncu"].get("runs", {}).values()):
    report.append("- NCU counters unavailable: ERR_NVGPUCTRPERM; PTXAS/static resource evidence remains archived.")
report += ["", "## Interpretation rule", "", "Retain only if both paired sessions improve prefill P50 and mean by at least 5%, P95 prefill latency and decode do not regress by more than 5%, exact greedy next-token parity against glproc and hardware correctness stay green, and W8PC remains spill-free in the >=24-warp resource tier.", "", "Diagnostic arithmetic and telemetry are not retention gates."]
(RESULTS / "WAVE7_REPORT.md").write_text("\n".join(report), encoding="utf-8")
archive = Path(shutil.make_archive(str(WORK / "glcuda_t4_ceiling_wave7_fetch_results"), "zip", root_dir=RESULTS))
print(f"Results directory: {RESULTS}")
print(f"Download archive: {archive}")
print(f"Archive size: {archive.stat().st_size / 1e6:.2f} MB")
print("\n" + "\n".join(report))
